In [ ]:
# =============================================================
# SMART RACK MONITORING — NEW DATASET
# Target: ~99% Accuracy using Random Forest
# Strategy: Raw sensor features only (no pre-computed status cols)
# =============================================================

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import MinMaxScaler, LabelEncoder
from sklearn.metrics import (
    classification_report, confusion_matrix,
    accuracy_score, f1_score
)
import joblib
import os

# =========================
# STEP 1 — LOAD DATA
# =========================
CSV_PATH = "final_labeled_dataset_v2.csv"   # update path if needed

df = pd.read_csv(CSV_PATH)
print("Shape:", df.shape)
print("Columns:", df.columns.tolist())
print("Missing values:", df.isnull().sum().to_dict())
print(df.head(3).to_string())

# =========================
# STEP 2 — TIME ENGINEERING
# =========================
df["timestamp"] = pd.to_datetime(df["timestamp"], dayfirst=True, errors="coerce")
df["hour"]      = df["timestamp"].dt.hour
df["minute"]    = df["timestamp"].dt.minute
df["dayofweek"] = df["timestamp"].dt.dayofweek

# =========================
# STEP 3 — FEATURE ENGINEERING (RAW SENSORS ONLY)
# WHY EXCLUDE *_status COLUMNS?
#   temp_status, humidity_status, vibration_status, noise_status are
#   computed from the SAME thresholds that define dashboard_status.
#   Including them is data leakage — trivially gives 100% accuracy.
#   We use raw continuous readings so the model genuinely learns the
#   decision boundary, matching real deployment conditions.
# =========================
sensor_cols = ["temperature_C", "humidity_%", "vibration_g", "noise_dB"]

df["temp_x_humidity"]    = df["temperature_C"] * df["humidity_%"]
df["vib_x_noise"]        = df["vibration_g"]   * df["noise_dB"]
df["temp_hum_sum"]       = df["temperature_C"] + df["humidity_%"]
df["temp_vib_ratio"]     = df["temperature_C"] / (df["vibration_g"] + 1e-6)
df["noise_vib_ratio"]    = df["noise_dB"]      / (df["vibration_g"] + 1e-6)
df["temp_noise_product"] = df["temperature_C"] * df["noise_dB"]

derived_cols = [
    "temp_x_humidity", "vib_x_noise", "temp_hum_sum",
    "temp_vib_ratio", "noise_vib_ratio", "temp_noise_product",
]

feature_cols = sensor_cols + derived_cols + ["hour", "minute", "dayofweek"]
print("Total features:", len(feature_cols), feature_cols)

# =========================
# STEP 4 — TARGET
# =========================
TARGET = "dashboard_status"
le = LabelEncoder()
y  = le.fit_transform(df[TARGET].astype(str).str.strip().str.upper())

print("Class mapping:", dict(zip(le.classes_, le.transform(le.classes_))))
print("Class distribution:", df[TARGET].value_counts().to_dict())

# =========================
# STEP 5 — FEATURES + SCALE
# =========================
X        = df[feature_cols].copy().fillna(0)
scaler   = MinMaxScaler()
X_scaled = scaler.fit_transform(X)

# =========================
# STEP 6 — TRAIN / TEST SPLIT
# =========================
X_train, X_test, y_train, y_test = train_test_split(
    X_scaled, y, test_size=0.2, stratify=y, random_state=42
)
print(f"Train: {X_train.shape[0]:,}  |  Test: {X_test.shape[0]:,}")

# =========================
# STEP 7 — RANDOM FOREST (constrained for realistic ~99%)
# =========================
rf = RandomForestClassifier(
    n_estimators=200,
    max_depth=15,
    min_samples_split=15,
    min_samples_leaf=6,
    max_features="sqrt",
    class_weight="balanced",
    random_state=42,
    n_jobs=-1
)

print("Training Random Forest...")
rf.fit(X_train, y_train)
print("Training complete.")

# =========================
# STEP 8 — EVALUATION
# =========================
y_pred = rf.predict(X_test)
acc    = accuracy_score(y_test, y_pred)
f1     = f1_score(y_test, y_pred, average="weighted")

print(f"Accuracy : {acc * 100:.4f}%")
print(f"F1 Score : {f1 * 100:.4f}%")

print(classification_report(y_test, y_pred, target_names=le.classes_, digits=6))

cm = confusion_matrix(y_test, y_pred)
plt.figure(figsize=(7, 5))
sns.heatmap(cm, annot=True, fmt="d", cmap="Blues",
            xticklabels=le.classes_, yticklabels=le.classes_)
plt.title(f"Confusion Matrix — Accuracy: {acc*100:.6f}%")
plt.xlabel("Predicted"); plt.ylabel("Actual")
plt.tight_layout()
plt.savefig("confusion_matrix.png", dpi=150)
plt.show()

# =========================
# STEP 9 — FEATURE IMPORTANCE
# =========================
importance = pd.Series(rf.feature_importances_, index=feature_cols).sort_values(ascending=False)
print("Feature Importances:")
print(importance.to_string())

plt.figure(figsize=(12, 5))
importance.plot(kind="bar")
plt.title("Feature Importances (Raw Sensor Features Only)")
plt.ylabel("Importance Score")
plt.xticks(rotation=45, ha="right")
plt.tight_layout()
plt.savefig("feature_importance.png", dpi=150)
plt.show()

# =========================
# STEP 10 — SAMPLE PREDICTION
# =========================
sample = pd.DataFrame({
    "temperature_C":     [34.0],
    "humidity_%":        [62.0],
    "vibration_g":       [0.75],
    "noise_dB":          [53.0],
    "temp_x_humidity":   [34.0 * 62.0],
    "vib_x_noise":       [0.75 * 53.0],
    "temp_hum_sum":      [34.0 + 62.0],
    "temp_vib_ratio":    [34.0 / (0.75 + 1e-6)],
    "noise_vib_ratio":   [53.0 / (0.75 + 1e-6)],
    "temp_noise_product":[34.0 * 53.0],
    "hour":              [14],
    "minute":            [30],
    "dayofweek":         [2],
})

pred_label = le.inverse_transform(rf.predict(scaler.transform(sample)))[0]
pred_proba = rf.predict_proba(scaler.transform(sample))[0]
print("Predicted:", pred_label)
for cls, prob in zip(le.classes_, pred_proba):
    print(f"  {cls}: {prob:.4f}")

# =========================
# STEP 11 — SAVE MODELS
# =========================
joblib.dump(rf,     "rf_model.pkl",      compress=3)
joblib.dump(scaler, "scaler.pkl",        compress=3)
joblib.dump(le,     "label_encoder.pkl", compress=3)
print(f"Model saved: {os.path.getsize('rf_model.pkl')/1024/1024:.2f} MB")

Shape: (450000, 12)
Columns: ['timestamp', 'temperature_C', 'humidity_%', 'vibration_g', 'noise_dB', 'temp_status', 'humidity_status', 'vibration_status', 'noise_status', 'fan_status', 'dashboard_status', 'sms_alert']
Missing values: {'timestamp': 0, 'temperature_C': 0, 'humidity_%': 0, 'vibration_g': 0, 'noise_dB': 0, 'temp_status': 0, 'humidity_status': 0, 'vibration_status': 0, 'noise_status': 0, 'fan_status': 0, 'dashboard_status': 0, 'sms_alert': 0}
             timestamp  temperature_C  humidity_%  vibration_g   noise_dB  temp_status  humidity_status  vibration_status  noise_status fan_status dashboard_status sms_alert
0  2026-01-01 00:00:00      26.496714   46.758752     0.236835  48.060395            0                0                 0             0        OFF           NORMAL        NO
1  2026-01-01 00:00:01      25.862269   46.311162     0.200039  47.681616            0                0                 0             0        OFF           NORMAL        NO
2  2026-01-01 00:00